In [ ]:
# =========================================================
# Cell 1 — Imports and Global Configuration
# =========================================================

import os
import csv
import gc
from datetime import datetime
import xml.etree.ElementTree as ET

import pandas as pd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

START_DATE = "2021-01-01"
END_DATE = "2023-12-31"

OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# Data Source Paths (UPDATED)
# ---------------------------------------------------------

PLATFORMS = {
    "ai": {
        "posts": r"D:\SO data\AI\Posts.xml",
        "comments": r"D:\SO data\AI\Comments.xml",
        "users": r"D:\SO data\AI\Users.xml",
    },

    "rus": {
        "posts": r"D:\SO data\rus.stackexchange.com\Posts.xml",
        "comments": r"D:\SO data\rus.stackexchange.com\Comments.xml",
        "users": r"D:\SO data\rus.stackexchange.com\Users.xml",
    }
}

print("Configuration loaded.")


In [ ]:
# =========================================================
# Cell 2 — Helper Functions
# =========================================================

def parse_date(date_str):
    """
    Convert Stack Exchange datetime string into Python datetime.

    Example:
    2021-01-01T12:34:56.789
    """

    if not date_str:
        return None

    try:
        return datetime.fromisoformat(date_str.replace("Z", ""))
    except Exception:
        try:
            return datetime.strptime(date_str[:19], "%Y-%m-%dT%H:%M:%S")
        except Exception:
            return None


def format_datetime(dt):
    """
    Convert datetime to YYYY-MM-DD HH:MM:SS
    """

    if dt is None:
        return None

    return dt.strftime("%Y-%m-%d %H:%M:%S")


def in_date_range(dt, start_dt, end_dt):
    """
    Check whether datetime falls in target range.
    """

    if dt is None:
        return False

    return start_dt <= dt <= end_dt


def safe_int(value):
    """
    Convert string to int safely.
    """

    if value in [None, ""]:
        return None

    try:
        return int(value)
    except:
        return None


def clear_element(elem):
    """
    Aggressively clear parsed XML element to save memory.
    """

    elem.clear()

    while elem.getprevious() is not None:
        del elem.getparent()[0]

In [ ]:
# =========================================================
# Cell 3 — Main Processing Function
# =========================================================

def process_platform(
    platform_name,
    posts_path,
    comments_path,
    users_path,
    start_date,
    end_date
):
    """
    Process one Stack Exchange platform.

    Parameters
    ----------
    platform_name : str
        "ai" or "rus"

    posts_path : str
        Path to Posts.xml

    comments_path : str
        Path to Comments.xml

    users_path : str
        Path to Users.xml

    start_date : str
        YYYY-MM-DD

    end_date : str
        YYYY-MM-DD
    """

    print("=" * 80)
    print(f"START PROCESSING PLATFORM: {platform_name}")
    print("=" * 80)

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")

    # =====================================================
    # Output Paths
    # =====================================================

    questions_csv = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_questions_raw.csv"
    )

    answers_csv = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_answers_raw.csv"
    )

    comments_csv = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_comments_raw.csv"
    )

    users_csv = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_users_raw.csv"
    )

    # =====================================================
    # Step 1 — Parse Users.xml
    # =====================================================

    print("\n[Step 1] Parsing Users.xml ...")

    user_dict = {}

    user_count = 0

    context = ET.iterparse(users_path, events=("end",))

    for event, elem in context:

        if elem.tag != "row":
            elem.clear()
            continue

        user_id = safe_int(elem.attrib.get("Id"))
        reputation = safe_int(elem.attrib.get("Reputation"))

        if user_id is not None:
            user_dict[user_id] = reputation

        user_count += 1

        if user_count % 100000 == 0:
            print(f"  Processed {user_count:,} users")

        elem.clear()

    del context
    gc.collect()

    print(f"Users loaded: {len(user_dict):,}")

    # =====================================================
    # Step 2 — First Pass Posts.xml
    # =====================================================

    print("\n[Step 2] First pass of Posts.xml ...")

    question_dict = {}
    accepted_answer_map = {}
    answer_to_question = {}

    user_ids_in_posts = set()

    question_counter = 0
    answer_counter = 0

    context = ET.iterparse(posts_path, events=("end",))

    for event, elem in context:

        if elem.tag != "row":
            elem.clear()
            continue

        attrib = elem.attrib

        post_type_id = attrib.get("PostTypeId")

        creation_dt = parse_date(attrib.get("CreationDate"))

        if not in_date_range(creation_dt, start_dt, end_dt):
            elem.clear()
            continue

        owner_user_id = safe_int(attrib.get("OwnerUserId"))

        if owner_user_id is not None:
            user_ids_in_posts.add(owner_user_id)

        # -------------------------------------------------
        # Questions
        # -------------------------------------------------

        if post_type_id == "1":

            question_id = safe_int(attrib.get("Id"))

            accepted_answer_id = safe_int(
                attrib.get("AcceptedAnswerId")
            )

            question_dict[question_id] = {
                "question_id": question_id,
                "creation_date": format_datetime(creation_dt),
                "body": attrib.get("Body"),
                "tags": attrib.get("Tags"),
                "owner_user_id": owner_user_id,
                "score": safe_int(attrib.get("Score")),
                "accepted_answer_id": accepted_answer_id,
                "post_type_id": 1
            }

            if accepted_answer_id is not None:
                accepted_answer_map[accepted_answer_id] = question_id

            question_counter += 1

            if question_counter % 10000 == 0:
                print(f"  Questions processed: {question_counter:,}")

        # -------------------------------------------------
        # Answers
        # -------------------------------------------------

        elif post_type_id == "2":

            answer_id = safe_int(attrib.get("Id"))
            parent_question_id = safe_int(attrib.get("ParentId"))

            if answer_id is not None and parent_question_id is not None:
                answer_to_question[answer_id] = parent_question_id

            answer_counter += 1

            if answer_counter % 50000 == 0:
                print(f"  Answers mapped: {answer_counter:,}")

        elem.clear()

    del context
    gc.collect()

    print(f"\nQuestions collected: {len(question_dict):,}")
    print(f"Answer mappings collected: {len(answer_to_question):,}")

    # =====================================================
    # Step 3 — Parse Comments.xml
    # =====================================================

    print("\n[Step 3] Parsing Comments.xml ...")

    user_ids_in_comments = set()

    comment_counter = 0

    with open(comments_csv, "w", newline="", encoding="utf-8") as f_comments:

        writer = csv.writer(f_comments)

        writer.writerow([
            "comment_id",
            "post_id",
            "user_id",
            "creation_date",
            "text",
            "parent_comment_id",
            "question_id"
        ])

        context = ET.iterparse(comments_path, events=("end",))

        for event, elem in context:

            if elem.tag != "row":
                elem.clear()
                continue

            attrib = elem.attrib

            creation_dt = parse_date(
                attrib.get("CreationDate")
            )

            if not in_date_range(
                creation_dt,
                start_dt,
                end_dt
            ):
                elem.clear()
                continue

            comment_id = safe_int(attrib.get("Id"))

            post_id = safe_int(attrib.get("PostId"))

            user_id = safe_int(attrib.get("UserId"))

            text = attrib.get("Text")

            # -------------------------------------------------
            # Infer question_id
            # -------------------------------------------------

            question_id = None

            if post_id in question_dict:
                question_id = post_id

            elif post_id in answer_to_question:
                question_id = answer_to_question[post_id]

            else:
                elem.clear()
                continue

            if user_id is not None:
                user_ids_in_comments.add(user_id)

            # -------------------------------------------------
            # Parent comment ID extraction
            # -------------------------------------------------

            parent_comment_id = None

            writer.writerow([
                comment_id,
                post_id,
                user_id,
                format_datetime(creation_dt),
                text,
                parent_comment_id,
                question_id
            ])

            comment_counter += 1

            if comment_counter % 100000 == 0:
                print(f"  Comments written: {comment_counter:,}")

            elem.clear()

        del context
        gc.collect()

    print(f"Comments written: {comment_counter:,}")

    # =====================================================
    # Step 4 — Write Questions CSV
    # =====================================================

    print("\n[Step 4] Writing questions CSV ...")

    with open(questions_csv, "w", newline="", encoding="utf-8") as f_questions:

        writer = csv.writer(f_questions)

        writer.writerow([
            "question_id",
            "creation_date",
            "body",
            "tags",
            "owner_user_id",
            "score",
            "accepted_answer_id",
            "post_type_id"
        ])

        for q in question_dict.values():

            writer.writerow([
                q["question_id"],
                q["creation_date"],
                q["body"],
                q["tags"],
                q["owner_user_id"],
                q["score"],
                q["accepted_answer_id"],
                q["post_type_id"]
            ])

    print(f"Questions written: {len(question_dict):,}")

    # =====================================================
    # Step 5 — Second Pass Posts.xml (Answers)
    # =====================================================

    print("\n[Step 5] Second pass Posts.xml for answers ...")

    answer_written = 0

    with open(answers_csv, "w", newline="", encoding="utf-8") as f_answers:

        writer = csv.writer(f_answers)

        writer.writerow([
            "answer_id",
            "parent_question_id",
            "creation_date",
            "body",
            "owner_user_id",
            "score",
            "is_accepted"
        ])

        context = ET.iterparse(posts_path, events=("end",))

        for event, elem in context:

            if elem.tag != "row":
                elem.clear()
                continue

            attrib = elem.attrib

            post_type_id = attrib.get("PostTypeId")

            if post_type_id != "2":
                elem.clear()
                continue

            creation_dt = parse_date(
                attrib.get("CreationDate")
            )

            if not in_date_range(
                creation_dt,
                start_dt,
                end_dt
            ):
                elem.clear()
                continue

            answer_id = safe_int(attrib.get("Id"))

            parent_question_id = safe_int(
                attrib.get("ParentId")
            )

            is_accepted = 0

            if answer_id in accepted_answer_map:
                is_accepted = 1

            writer.writerow([
                answer_id,
                parent_question_id,
                format_datetime(creation_dt),
                attrib.get("Body"),
                safe_int(attrib.get("OwnerUserId")),
                safe_int(attrib.get("Score")),
                is_accepted
            ])

            answer_written += 1

            if answer_written % 50000 == 0:
                print(f"  Answers written: {answer_written:,}")

            elem.clear()

        del context
        gc.collect()

    print(f"Answers written: {answer_written:,}")

    # =====================================================
    # Step 6 — Write Users CSV
    # =====================================================

    print("\n[Step 6] Writing users CSV ...")

    all_user_ids = (
        user_ids_in_posts
        .union(user_ids_in_comments)
    )

    user_written = 0

    with open(users_csv, "w", newline="", encoding="utf-8") as f_users:

        writer = csv.writer(f_users)

        writer.writerow([
            "user_id",
            "reputation"
        ])

        for user_id in all_user_ids:

            if user_id is None:
                continue

            writer.writerow([
                user_id,
                user_dict.get(user_id)
            ])

            user_written += 1

    print(f"Users written: {user_written:,}")

    # =====================================================
    # Step 7 — Sample Dataset Generation
    # =====================================================

    print("\n[Step 7] Creating sample datasets ...")

    q_df = pd.read_csv(questions_csv)

    sample_questions = q_df.head(1000)

    sample_question_ids = set(
        sample_questions["question_id"].tolist()
    )

    sample_questions.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"sample_{platform_name}_questions_raw.csv"
        ),
        index=False
    )

    # -----------------------------------------------------
    # Answers sample
    # -----------------------------------------------------

    a_df = pd.read_csv(answers_csv)

    sample_answers = a_df[
        a_df["parent_question_id"]
        .isin(sample_question_ids)
    ]

    sample_answers.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"sample_{platform_name}_answers_raw.csv"
        ),
        index=False
    )

    # -----------------------------------------------------
    # Comments sample
    # -----------------------------------------------------

    c_df = pd.read_csv(comments_csv)

    sample_comments = c_df[
        c_df["question_id"]
        .isin(sample_question_ids)
    ]

    sample_comments.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"sample_{platform_name}_comments_raw.csv"
        ),
        index=False
    )

    # -----------------------------------------------------
    # Users sample
    # -----------------------------------------------------

    sample_user_ids = set()

    sample_user_ids.update(
        sample_questions["owner_user_id"]
        .dropna()
        .astype(int)
        .tolist()
    )

    sample_user_ids.update(
        sample_answers["owner_user_id"]
        .dropna()
        .astype(int)
        .tolist()
    )

    sample_user_ids.update(
        sample_comments["user_id"]
        .dropna()
        .astype(int)
        .tolist()
    )

    u_df = pd.read_csv(users_csv)

    sample_users = u_df[
        u_df["user_id"]
        .isin(sample_user_ids)
    ]

    sample_users.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"sample_{platform_name}_users_raw.csv"
        ),
        index=False
    )

    print("Sample datasets created.")

    # =====================================================
    # Summary
    # =====================================================

    summary = {
        "platform": platform_name,
        "questions": len(q_df),
        "answers": len(a_df),
        "comments": len(c_df),
        "users": len(u_df)
    }

    print("\nPROCESSING COMPLETE")
    print(summary)

    return summary

In [ ]:
# =========================================================
# Cell 4 — Run Processing for All Platforms
# =========================================================

summaries = []

for platform_name, paths in PLATFORMS.items():

    summary = process_platform(
        platform_name=platform_name,
        posts_path=paths["posts"],
        comments_path=paths["comments"],
        users_path=paths["users"],
        start_date=START_DATE,
        end_date=END_DATE
    )

    summaries.append(summary)

print("\nAll platforms processed.")

In [ ]:
# =========================================================
# Cell 5 — Summary Statistics
# =========================================================

summary_df = pd.DataFrame(summaries)

print(summary_df)

In [ ]:
# =========================================================
# Cell 6 — Display First 3 Rows of Each CSV
# =========================================================

for platform_name in ["ai", "rus"]:

    print("\n" + "=" * 80)
    print(f"PLATFORM: {platform_name}")
    print("=" * 80)

    for table_name in [
        "questions",
        "answers",
        "comments",
        "users"
    ]:

        file_path = os.path.join(
            OUTPUT_DIR,
            f"{platform_name}_{table_name}_raw.csv"
        )

        print(f"\n{table_name.upper()}")

        df = pd.read_csv(file_path)

        display(df.head(3))

In [ ]:
# =========================================================
# Cell 7 — Validation Checks
# =========================================================

for platform_name in ["ai", "rus"]:

    print("\n" + "=" * 80)
    print(f"VALIDATION — {platform_name}")
    print("=" * 80)

    q_path = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_questions_raw.csv"
    )

    a_path = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_answers_raw.csv"
    )

    c_path = os.path.join(
        OUTPUT_DIR,
        f"{platform_name}_comments_raw.csv"
    )

    # -----------------------------------------------------
    # Load Data
    # -----------------------------------------------------

    q_df = pd.read_csv(q_path)
    a_df = pd.read_csv(a_path)
    c_df = pd.read_csv(c_path)

    # -----------------------------------------------------
    # Date Range Validation
    # -----------------------------------------------------

    print("\nQuestions Date Range:")
    print(q_df["creation_date"].min())
    print(q_df["creation_date"].max())

    print("\nAnswers Date Range:")
    print(a_df["creation_date"].min())
    print(a_df["creation_date"].max())

    print("\nComments Date Range:")
    print(c_df["creation_date"].min())
    print(c_df["creation_date"].max())

    # -----------------------------------------------------
    # Missing question_id in comments
    # -----------------------------------------------------

    missing_question_ids = c_df["question_id"].isna().sum()

    print(f"\nMissing comment question_id: {missing_question_ids}")

    # -----------------------------------------------------
    # Negative Delay Check
    # -----------------------------------------------------

    q_times = dict(
        zip(
            q_df["question_id"],
            pd.to_datetime(q_df["creation_date"])
        )
    )

    negative_delay_count = 0

    c_df["creation_date"] = pd.to_datetime(
        c_df["creation_date"]
    )

    for idx, row in c_df.iterrows():

        qid = row["question_id"]

        if qid not in q_times:
            continue

        if row["creation_date"] < q_times[qid]:
            negative_delay_count += 1

    print(
        f"\nNegative delay comments count: "
        f"{negative_delay_count}"
    )

    # -----------------------------------------------------
    # Accepted Answer Validation
    # -----------------------------------------------------

    accepted_answers = a_df[
        a_df["is_accepted"] == 1
    ].head(5)

    print("\nSample accepted answers:")
    display(accepted_answers)

In [ ]:
# =========================================================
# Cell 8 — File Size Report
# =========================================================

print("=" * 80)
print("OUTPUT FILE SIZES")
print("=" * 80)

for filename in os.listdir(OUTPUT_DIR):

    full_path = os.path.join(OUTPUT_DIR, filename)

    size_mb = os.path.getsize(full_path) / (1024 * 1024)

    print(f"{filename:<45} {size_mb:>10.2f} MB")